<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 16 · 让使用反馈进入下一版 Skill

金额精度经验可以生成操作步骤，但第一次用到新项目时，仍可能发现遗漏。这次让真实 Agent 读取 Skill、检查一个有缺陷的函数，把失败作为使用反馈，再让模型提出下一版 Skill，审核后交给新的 Agent 执行。

需要真实 Generation 和支持工具调用的对话模型。两个 Agent 在不同 Python 进程中运行，共用本篇项目文件，不共享会话历史。它们执行的工具源码在 [support/amount_agent.py](support/amount_agent.py)，测试素材在 [support/test_amount_fixture.txt](support/test_amount_fixture.txt)。

路线：已确认经验 → 生成 Skill → 第一次真实使用 → 记录反馈 → 改进候选 → 审核 → 第二次使用。两次执行展示反馈链路，不用于推断 Skill 的普遍效果提升。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("16", features=("generation",))
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 16", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 从有边界的经验生成 Skill

先实际验证三位小数的精度问题，再保存这条有限经验。Skill 生成选择 experience 来源，引用确切 Experience 版本；不是再次把同样文字当 Source 生成。

In [ ]:
from decimal import Decimal

from powercontext.http import (
    ApproveArtifactCandidateRequest,
    ArtifactReference,
    CreateArtifactRequest,
    GenerateSkillRequest,
)

assert int(Decimal("1.999") * 100) == 199
experience = await client.create_artifact(
    scope_id,
    CreateArtifactRequest.model_validate({
        "family": "experience",
        "content": {
            "situation": "amount 直接转整数分会静默截断 1.999",
            "action": "转换之前检查是否为整数分",
            "outcome": "本单元格确认旧写法得到 199",
            "lesson": "Skill name csv-amount-feedback: check precision before integer conversion; this evidence only covers fractional cents.",
        },
    }),
)
experience_ref = ArtifactReference(
    family="experience", artifact_id=experience.artifact_id, revision=experience.revision
)
generated = await client.generate_skill(
    GenerateSkillRequest(
        scope_id=scope_id,
        origin="experience",
        source_refs=[],
        artifact_refs=[experience_ref],
        reason="Generate a Skill named csv-amount-feedback. Explain actual checks, and do not claim unexecuted cases passed.",
    )
)
assert generated.candidate and generated.candidate.status == "pending"
show(generated.candidate.proposal)
approved = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id, candidate_id=generated.candidate.candidate_id, expected_version=generated.candidate.version
    )
)
assert approved.result_artifact
skill_ref = approved.result_artifact

## 准备可失败的真实项目

实现使用直接整数转换，测试包括正常值、超精度、负数和非有限值。首个 Agent 只有检查权限，必须读取 Skill 并运行测试，不能悄悄修改实现。

In [ ]:
import asyncio
import base64
import io
import json
import zipfile

from powercontext.http import GetSkillPackageRequest

support = Path(sys.modules["_tutorial"].__file__).parent / "support"
project = lab.directory / "project"
project.mkdir()
(project / "amount.py").write_text(
    "from decimal import Decimal\ndef cents(text):\n    return int(Decimal(text) * 100)\n", encoding="utf-8"
)
(project / "test_amount.py").write_bytes((support / "test_amount_fixture.txt").read_bytes())
request = GetSkillPackageRequest(scope_id=scope_id, artifact=skill_ref)
package = await client.download_skill_package(request)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(package.archive_base64))) as archive:
    selected_skill = project / "SKILL.md"
    selected_skill.write_bytes(archive.read("SKILL.md"))


async def run_agent(task, can_edit):
    process = await asyncio.create_subprocess_exec(
        sys.executable,
        str(support / "amount_agent.py"),
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    payload = {"workspace": str(project), "skill_path": str(selected_skill), "task": task, "can_edit": can_edit}
    try:
        stdout, _stderr = await asyncio.wait_for(process.communicate(json.dumps(payload).encode()), timeout=180)
    except TimeoutError:
        process.kill()
        await process.wait()
        raise
    assert process.returncode == 0, "Agent 子进程失败；请检查模型工具调用能力"
    return json.loads(stdout)


first = await run_agent(
    "按所选 Skill 检查当前金额函数。必须 inspect_project 和 run_checks。只诊断，指出有限值、负数、精度和非法文本的实际结果，不修改文件。",
    False,
)
check_messages = [
    message for message in first["messages"] if message["type"] == "tool" and message.get("name") == "run_checks"
]
assert check_messages and json.loads(check_messages[-1]["content"])["exit_code"] != 0
print(first["answer"])

## 把执行结果与准确 Skill 版本连起来

使用记录必须指向实际使用的版本和包摘要。task_source 保存真实输出，usage 记录选择、调用、验证与结果，避免把“被推荐过”写成“执行成功”。

In [ ]:
from powercontext.http import CreateSourceRequest, RecordSkillUsageRequest, SourceReference

work = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content={"agent_pid": first["pid"], "actual_checks": check_messages, "finding": first["answer"]}
    ),
)
usage = await client.record_skill_usage(
    RecordSkillUsageRequest(
        scope_id=scope_id,
        observation_id=f"{lab.run_id}:first",
        skill_ref=skill_ref,
        package_digest="sha256:" + package.package.tree_digest,
        target_id="tutorial-agent",
        selected=True,
        invoked="true",
        validation="failed",
        outcome="failure",
        task_source=SourceReference(name="content", source_id=work.source_id),
    )
)
show({"使用版本": skill_ref.model_dump(), "使用证据": usage.source.model_dump(), "结果": "failure"})

## 生成替换候选，审核后才形成新版本

usage 来源必须同时指定被改进的 Skill 和反馈证据。先检查 pending 时旧版本未变化，再审核新候选。这里的自动批准只用于显示过内容、且执行范围受测试约束的教学实验。

In [ ]:
revision = await client.generate_skill(
    GenerateSkillRequest(
        scope_id=scope_id,
        origin="usage",
        target=skill_ref,
        source_refs=[usage.source, SourceReference(name="content", source_id=work.source_id)],
        artifact_refs=[skill_ref],
        reason="Keep the same valid Skill name. Add explicit negative, non-finite, invalid text and fractional-cent rejection checks from the actual failures. Require ValueError for invalid input. Do not claim the implementation is fixed.",
    )
)
assert revision.candidate and revision.candidate.status == "pending"
assert (await client.get_artifact(scope_id, "skill", skill_ref.artifact_id)).revision == skill_ref.revision
show(revision.candidate.proposal)
new_approval = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id, candidate_id=revision.candidate.candidate_id, expected_version=revision.candidate.version
    )
)
assert new_approval.result_artifact and new_approval.result_artifact.artifact_id == skill_ref.artifact_id
new_ref = new_approval.result_artifact
new_package = await client.download_skill_package(GetSkillPackageRequest(scope_id=scope_id, artifact=new_ref))
with zipfile.ZipFile(io.BytesIO(base64.b64decode(new_package.archive_base64))) as archive:
    selected_skill.write_bytes(archive.read("SKILL.md"))

## 新 Agent 按新版本执行，并独立验收

现在明确授权第二个 Agent 只修改 amount.py。Notebook 最后再次启动测试进程，结果不能只依赖 Agent 的口头总结。我们同时记录第二次真实使用。

In [ ]:
second = await run_agent(
    "读取当前 Skill 和测试，修复 cents(text)。正常金额转整数分；负数、非有限值、超精度、非法文本必须抛 ValueError。只修改 amount.py，运行测试直到通过，报告真实结果。",
    True,
)
assert second["pid"] != first["pid"]
process = await asyncio.create_subprocess_exec(
    sys.executable,
    "-m",
    "unittest",
    "-v",
    "test_amount",
    cwd=project,
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.PIPE,
)
stdout, stderr = await asyncio.wait_for(process.communicate(), 30)
assert process.returncode == 0, (stdout + stderr).decode()
print((stdout + stderr).decode())
verified = await client.create_source(
    scope_id,
    CreateSourceRequest(
        content={
            "actual_exit_code": process.returncode,
            "checks": (stdout + stderr).decode(),
            "agent_pid": second["pid"],
        }
    ),
)
await client.record_skill_usage(
    RecordSkillUsageRequest(
        scope_id=scope_id,
        observation_id=f"{lab.run_id}:second",
        skill_ref=new_ref,
        package_digest="sha256:" + new_package.package.tree_digest,
        target_id="tutorial-agent",
        selected=True,
        invoked="true",
        validation="passed",
        outcome="success",
        task_source=SourceReference(name="content", source_id=verified.source_id),
    )
)
table([
    {"使用": "第一次", "Skill Revision": skill_ref.revision, "实际测试": "失败"},
    {"使用": "第二次", "Skill Revision": new_ref.revision, "实际测试": "通过"},
])

## 练习与验收

查看两个 Agent 的工具消息，确认都实际读取了 SKILL.md 并运行测试。再增加一个明确的新需求，观察它如何进入新的使用记录和候选，而不是直接覆盖旧 Skill。

接下来阅读 [17_external_skill_packages.ipynb](17_external_skill_packages.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")